# Advanced Topics

This notebook covers power-user features and best practices:

- Working with **UUIDs** for element identification
- **IP addressing** and static routing tables
- Direct **DataFrame manipulation**
- The **connector** pattern
- Best practices for large networks

## Setup

In [1]:
import pandaict as pi
from pandaict.utils.enums import NodeType, DeviceType, AppType, InfoItem
import uuid

## UUID-based element identification

By default, pandaict uses index tuples like `('node', 0)`. When you need globally unique, serialization-safe identifiers, enable UUID mode with `use_uuid=True`.

In [2]:
net = pi.create_empty_network(name='UUID Demo')

# Create elements with UUID return values
uid_node = pi.create_ict_node(net, name='UUID-Node', node_type=NodeType.ROUTER,
                              link_types=['LAN1000'], use_uuid=True)
uid_dev  = pi.create_ict_device(net, name='UUID-Dev', device_type=DeviceType.SERVER,
                                use_uuid=True)

print('Node UUID:', uid_node)
print('Dev UUID: ', uid_dev)
print('Type:     ', type(uid_node).__name__)

Node UUID: a4161b22-c8c2-4124-bc63-0898c2594887
Dev UUID:  b2596b94-16c1-4302-8588-03015bf025fa
Type:      UUID


In [3]:
# You can pass a specific UUID
my_uid = uuid.uuid4()
uid_node2 = pi.create_ict_node(net, name='Custom-UID-Node',
                               node_type=NodeType.ROUTER,
                               link_types=['LAN1000'],
                               use_uuid=True, uid=my_uid)
print('Matches:', uid_node2 == my_uid)

Matches: True


In [4]:
# Create a link using UUID references
pi.create_ict_link(net, name='UUID-Link', from_elt=uid_node, to_elt=[uid_dev],
                   link_type='LAN1000', use_uuid=True)

print('UUIDs stored in link DataFrame:')
print('  from:', type(net.link.loc[0, 'from']).__name__, '->', net.link.loc[0, 'from'])
print('  to:  ', type(net.link.loc[0, 'to'][0]).__name__, '->', net.link.loc[0, 'to'][0])

UUIDs stored in link DataFrame:
  from: tuple -> ('node', np.int64(0))
  to:   tuple -> ('device', np.int64(0))


## IP addressing

The `topology` package provides IP address management and static routing table support:

In [5]:
# Rebuild a clean network for IP demos
net2 = pi.create_empty_network(name='IP Demo')

n1 = pi.create_ict_node(net2, name='R1', node_type=NodeType.ROUTER, link_types=['LAN1000'])
n2 = pi.create_ict_node(net2, name='R2', node_type=NodeType.ROUTER, link_types=['LAN1000'])
d1 = pi.create_ict_device(net2, name='Host1', device_type=DeviceType.SERVER)
d2 = pi.create_ict_device(net2, name='Host2', device_type=DeviceType.SERVER)

pi.create_ict_link(net2, name='L-R1R2', from_elt=n1, to_elt=[n2], link_type='LAN1000')
pi.create_ict_link(net2, name='L-H1R1', from_elt=d1, to_elt=[n1], link_type='LAN1000')
pi.create_ict_link(net2, name='L-H2R2', from_elt=d2, to_elt=[n2], link_type='LAN1000')

print('IP demo network ready.')

IP demo network ready.


In [6]:
from pandaict.topology.ip_address import IP_address
from pandaict.topology.ip_support import (
    assign_ipaddr_to_elt, assign_ipaddr_to_elts,
    set_static_iptable_entry, update_iptables
)

# Assign IPs to devices (CIDR notation)
assign_ipaddr_to_elt(net2, d1, '192.168.1.10', netmask='255.255.255.0')
assign_ipaddr_to_elt(net2, d2, '192.168.2.10', netmask='255.255.255.0')

print('Device IPs:')
for _, dev_row in net2.device.iterrows():
    print(f"  {dev_row['name']}: {dev_row['ip_addr']}")

Device IPs:
  Host1: 192.168.1.10
  Host2: 192.168.2.10


**Static routing**: Once IPs are assigned and connections are defined, you can use
`set_static_iptable_entry()` to configure static routes, and `update_iptables()` to
auto-derive node indices. Pass `respect_iptables=True` to Monte Carlo reliability
functions to enforce these routing rules.

## Direct DataFrame manipulation

You can use standard pandas operations to inspect and modify the element DataFrames. This is powerful but requires care — some internal consistency rules are enforced at creation time but not on direct mutation.

In [9]:
# Filter nodes with pandas
router_nodes = net2.node[net2.node['type'] == NodeType.ROUTER]
print('Router nodes:')
display(router_nodes[['name', 'type']])

Router nodes:


,name,type
0,R1,NodeType.ROUTER
1,R2,NodeType.ROUTER


In [10]:
# Bulk-update in_service field
net2.link['in_service'] = True
print('All links in_service:', all(net2.link['in_service']))

All links in_service: True


In [11]:
# Add a custom column (not used by pandaict, but may be useful for your own tracking)
net2.device['location'] = 'Building A'
net2.device.at[d1[1], 'location'] = 'Building B'
display(net2.device[['name', 'location']])

,name,location
0,Host1,Building B
1,Host2,Building A


**Caution**: Direct DataFrame mutation can break internal consistency (e.g., changing an element type from `ROUTER` to `SWITCH` after links are created). Prefer the `create_*()` API functions when possible.

## Connectors

pandaict supports a **connector** pattern for domain-specific extensions. Connectors are Python packages that, when imported, register:

- Additional standard types (link and protocol specs)
- Custom enums (device types, info items)
- Specialized utility functions

The reference connector is `pp_pict_connector` (pandapower-pandaict), which adds IEC 61850 protocol support, electrical meter value info items, and power-system-specific element types.

The global list `connectors_installed` in `pandaict/__init__.py` tracks which connectors have been loaded.

In [12]:
from pandaict import connectors_installed
print('Connectors installed:', connectors_installed)
print('(Empty unless a connector package was imported)')

Connectors installed: []
(Empty unless a connector package was imported)


In this case no connector was installed beforehand. A fleshed out tutorial focused on the domain specific connectors will follow in the future

## Best practices

### Numba
- The Monte Carlo engine uses Numba JIT compilation. The **first call** compiles kernels and may take a few seconds. Subsequent calls are fast.
- Numba is optional; the engine falls back to pure Python (`use_numba=False`) if Numba is unavailable. Still Numba usage is highly recommended since it speeds up the calculation by orders of magnitude and always contains the latest features and fixes.

### Large networks
- Network files can grow large; use `to_excel()` / `from_excel()` for fast serialization.

### Debugging
- Enable verbose inspection: `pi.inspect(net, verbose=True)`.
- Check `pandaICT_Core.log` for detailed routing/reliability debug output.
- For type errors, verify that enum values match the expected categories (e.g., node_type must be a `NodeType`, not an `AppType`).

## Where to go next

- **[Reliability Monte Carlo series](08_how_to_MTBF_estimation.ipynb)**: deep dive into stochastic reliability: MTTF/MTTR estimation, critical component analysis, empirical reliability functions
- **[pp_pict_connector example](../pp_pict_connector_example.ipynb)**: pandapower-pandaict connector for power systems modeling
- **[pandaICT datamodel](../pandaICT_datamodel.ipynb)**: formal element reference
- **[pandaICT routing](../pandaICT_routing.ipynb)**: advanced routing topics
- Official Sphinx Documentation

This concludes the how-to series. Thanks for following along!